In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'openset'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    df =pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)==50)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_5": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

100%|██████████| 5217/5217 [00:00<00:00, 36031.75it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,metric,value
80,0.1,0.25,20NG,ADB,0,ACC,53.8000
81,0.1,0.25,20NG,ADB,0,F1,50.7301
82,0.1,0.25,20NG,ADB,0,K-F1,48.3731
83,0.1,0.25,20NG,ADB,0,N-F1,62.5150
148,0.1,0.50,20NG,ADB,0,ACC,60.3900
...,...,...,...,...,...,...,...
16919,1.0,0.50,thucnews,clap,4,N-F1,0.0000
16932,1.0,0.75,thucnews,clap,4,ACC,61.2200
16933,1.0,0.75,thucnews,clap,4,F1,65.5622
16934,1.0,0.75,thucnews,clap,4,K-F1,0.0000


In [2]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio               0.1                                             \
known_cls_ratio            0.25                              0.50            
metric                      ACC       F1     K-F1     N-F1    ACC       F1   
dataset  method fold_idx                                                     
20NG     ADB    0         53.80  50.7301  48.3731  62.5150  60.39  69.6270   
                1         40.82  40.1574  39.6174  42.8571  66.04  67.4986   
                2         22.19  25.5297  27.3626  16.3651  48.42  57.5567   
                3         51.45  56.4627  56.9910  53.8210  62.88  69.5776   
                4         58.17  49.3235  45.7212  67.3349  80.50  81.5961   
...                         ...      ...      ...      ...    ...      ...   
thucnews clap   0         22.94  32.5598   0.0000   0.0000  35.19  43.5344   
                1         22.80  30.2252   0.0000   0.0000  43.35  48.8931   
                2         37.30  42.1816   0.0000   0.0000  35.75  42.8826   
                3         23.15  30.2534   0.0000   0.0000  35.47  42.5360   
                4         22.73  21.0056   0.0000   0.0000  22.52  22.7761   

labeled_ratio                                               ...      1.0  \
known_cls_ratio                              0.75           ...     0.25   
metric                       K-F1     N-F1    ACC       F1  ...     K-F1   
dataset  method fold_idx                                    ...            
20NG     ADB    0         71.6784  49.1130  75.92  82.8313  ...  67.5778   
                1         67.7088  65.3968  75.99  82.3294  ...  74.2382   
                2         60.1148  31.9749  73.44  80.9509  ...  61.4170   
                3         71.0790  54.5631  94.22  95.7202  ...  67.2164   
                4         81.6761  80.7968  88.90  90.8816  ...  75.9690   
...                           ...      ...    ...      ...  ...      ...   
thucnews clap   0          0.0000   0.0000  50.32  54.4698  ...   0.0000   
                1          0.0000   0.0000  47.08  50.7528  ...   0.0000   
                2          0.0000   0.0000  46.73  49.9383  ...   0.0000   
                3          0.0000   0.0000  47.92  51.6253  ...   0.0000   
                4          0.0000   0.0000  44.48  42.1609  ...   0.0000   

labeled_ratio                                                               \
known_cls_ratio                     0.50                              0.75   
metric                       N-F1    ACC       F1     K-F1     N-F1    ACC   
dataset  method fold_idx                                                     
20NG     ADB    0         69.8795  76.26  81.8210  82.8430  71.6010  86.95   
                1         80.5954  86.62  89.4433  89.8371  85.5051  80.50   
                2         56.7916  75.05  81.3614  82.7434  67.5416  90.32   
                3         66.7866  85.88  88.5902  89.1119  83.3731  96.50   
                4         85.0974  87.90  90.5715  91.0527  85.7595  91.19   
...                           ...    ...      ...      ...      ...    ...   
thucnews clap   0          0.0000  51.79  60.0603   0.0000   0.0000  64.11   
                1          0.0000  51.30  57.9459   0.0000   0.0000  61.08   
                2          0.0000  47.99  53.8794   0.0000   0.0000  64.04   
                3          0.0000  50.53  56.3245   0.0000   0.0000  62.70   
                4          0.0000  45.11  51.6585   0.0000   0.0000  61.22   

labeled_ratio                                        
known_cls_ratio                                      
metric                         F1     K-F1     N-F1  
dataset  method fold_idx                             
20NG     ADB    0         91.9988  93.4491  70.2454  
                1         87.5089  90.8249  37.7682  
                2         94.2404  95.3900  76.9968  
                3         97.5276  97.9113  91.7722  
                4         93.7982  94.8427  78.1302  
...                           .

In [3]:
# 基于 df_melted = [labeled_ratio, known_cls_ratio, dataset, method, fold_idx, metric, value]

# 认为：只要某个 metric 有记录，就视为该 (dataset, method, labeled_ratio, known_cls_ratio, fold_idx) 已测试
cells = (df_melted[["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"]]
         .drop_duplicates()
         .sort_values(["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"])
         .reset_index(drop=True))

# 1) 每个 dataset × method 的维度覆盖与计数
progress_by_dm = (cells
    .groupby(["dataset","method"])
    .agg(
        labeled_ratios=("labeled_ratio", lambda s: sorted(pd.unique(s))),
        n_labeled=("labeled_ratio", pd.Series.nunique),
        known_cls_ratios=("known_cls_ratio", lambda s: sorted(pd.unique(s))),
        n_known=("known_cls_ratio", pd.Series.nunique),
        folds=("fold_idx", lambda s: sorted(pd.unique(s))),
        n_folds=("fold_idx", pd.Series.nunique),
        n_cells=("fold_idx", "size"),  # 完成的组合总数
    )
    .reset_index()
)

# 2) 每个 dataset × method × labeled_ratio × known_cls_ratio 下完成了多少个 fold
fold_coverage = (cells
    .groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
    .nunique()
    .reset_index(name="folds_done")
    .sort_values(["dataset","method","labeled_ratio","known_cls_ratio"])
)

In [4]:
# ====== 2) （可选）若有计划 fold 列表，显示“完成数/计划数” ======
# 例如计划 folds 为 [0,1,2,3,4]（共5个）
EXPECTED_FOLDS = None  # e.g., [0,1,2,3,4]
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)
    # 统计每格完成的 fold 个数
    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text"
    ).sort_index()

# ====== 3) 每格显示“已完成的 fold 列表”，便于直观看缺哪几个 ======
# 先做一个按格聚合出 fold 列表的表
fold_list = (cells.groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
             .apply(lambda s: sorted(pd.unique(s)))
             .reset_index(name="folds_list"))
fold_list["folds_list_str"] = fold_list["folds_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")

pivot_folds_list = fold_list.pivot(
    index=["dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str"
).sort_index()

pivot_folds_list.to_excel(f'results/{task}_progress.xlsx')
pivot_folds_list

labeled_ratio                 0.1                                    \
known_cls_ratio              0.25             0.50             0.75   
dataset  method                                                       
20NG     ADB      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         DOC      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         DeepUnk  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         DyEn     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                           ...              ...              ...   
thucnews KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         PLM_OOD  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         SCL      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         ab       [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         clap     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   

labeled_ratio                 0.5                                    \
known_cls_ratio              0.25             0.50             0.75   
dataset  method                                                       
20NG     ADB      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         DOC      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         DeepUnk     [0, 1, 2, 3]        [0, 1, 2]              [0]   
         DyEn     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                           ...              ...              ...   
thucnews KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         PLM_OOD  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         SCL      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         ab       [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
         clap     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   

labeled_ratio                 1.0                                    
known_cls_ratio              0.25             0.50             0.75  
dataset  method                                                      
20NG     ADB      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         DOC      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         DeepUnk     [0, 1, 2, 3]        [0, 1, 2]              [0]  
         DyEn     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
...                           ...              ...              ...  
thucnews KnnCon   [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         PLM_OOD  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         SCL      [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         ab       [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  
         clap     [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]  

[108 rows x 9 columns]

In [5]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.9886831275720165

In [6]:
all_num

4860

In [7]:
cur_num

4805.0